# Running evaluations on Isambard

Evaluate **gpt-oss-120b** with the UK AISI
[Inspect](https://inspect.aisi.org.uk/) framework, against a vLLM server
running on an Isambard GH200 node.

**Start the server first**, from a login node:

```bash
bash serve_vllm.sh
```

It serves the model as an OpenAI-compatible endpoint and writes its URL to
`endpoint.txt`. Select the **`.venv`** kernel for this notebook. Everything below talks HTTP, so the notebook never touches a
GPU -- the ~4-minute model load is paid once by the server, and you iterate
against it interactively.

Based on the official
[distributed-inference tutorial](https://docs.isambard.ac.uk/user-documentation/tutorials/distributed-inference/).


## 1. Connect to the server

In [ ]:
import os, pathlib, urllib.request, json, logging, warnings

# Inspect runs an async event loop; under Jupyter its teardown emits harmless
# noise ("Task was destroyed", "coroutine was never awaited"). Quieten it.
logging.getLogger("asyncio").setLevel(logging.CRITICAL)
warnings.filterwarnings("ignore", message="coroutine .* was never awaited")

# serve_vllm.sh writes this when the server starts.
endpoint_file = pathlib.Path("endpoint.txt")
if endpoint_file.exists():
    ENDPOINT = endpoint_file.read_text().strip()
else:
    # Fall back to a manual value, e.g. if you started the server yourself.
    ENDPOINT = os.environ.get("ISAMBARD_BASE_URL", "http://localhost:8000/v1")

MODEL_NAME = os.environ.get("SERVED_NAME", "gpt-oss-120b")

# Inspect resolves `openai-api/<service>/<model>` by reading <SERVICE>_BASE_URL
# and <SERVICE>_API_KEY. The key is unused by vLLM but must be non-empty.
os.environ["ISAMBARD_BASE_URL"] = ENDPOINT
os.environ["ISAMBARD_API_KEY"] = "dummy"

print(f"endpoint : {ENDPOINT}")
print(f"model    : {MODEL_NAME}")

with urllib.request.urlopen(ENDPOINT + "/models", timeout=30) as r:
    served = [m["id"] for m in json.load(r)["data"]]
print(f"serving  : {served}")
assert MODEL_NAME in served, f"{MODEL_NAME} not served; found {served}"

# A simple math evaluation with AISI Inspect

GSM8K is grade-school maths word problems -- small, fast, and a reasonable
smoke test that the whole chain works. An Inspect **task** is just three
pieces: a **dataset**, a **solver** (here, plain `generate()`), and a
**scorer**. We define a small GSM8K task inline rather than importing one, so
you can see all the moving parts -- and so this notebook needs only
`inspect-ai`, not the heavier `inspect-evals` benchmark library.

`limit` keeps this interactive; drop it for the full 1319-problem test split.


In [ ]:
import re
from inspect_ai import Task, task, eval as inspect_eval
from inspect_ai.dataset import hf_dataset, Sample
from inspect_ai.solver import generate
from inspect_ai.scorer import scorer, accuracy, stderr, Score, Target, CORRECT, INCORRECT

MODEL = f"openai-api/isambard/{MODEL_NAME}"

def record_to_sample(record):
    # GSM8K gold answers end with "#### <final number>".
    target = record["answer"].split("####")[-1].strip().replace(",", "")
    return Sample(input=record["question"], target=target)

def last_number(text):
    nums = re.findall(r"-?\d[\d,]*(?:\.\d+)?", text.replace(",", ""))
    return nums[-1] if nums else None

@scorer(metrics=[accuracy(), stderr()])
def gsm8k_numeric():
    async def score(state, target: Target):
        pred = last_number(state.output.completion)
        ok = pred is not None and abs(float(pred) - float(target.text)) < 1e-6
        return Score(value=CORRECT if ok else INCORRECT, answer=pred)
    return score

@task
def gsm8k(limit=10):
    return Task(
        dataset=hf_dataset("openai/gsm8k", split="test", name="main",
                           sample_fields=record_to_sample, limit=limit),
        solver=generate(),
        scorer=gsm8k_numeric(),
    )

logs = inspect_eval(gsm8k(limit=10), model=MODEL, max_connections=10, display="plain")
log = logs[0]
print(f"\nstatus  : {log.status}")
print(f"samples : {len(log.samples or [])}")


### The headline numbers

In [ ]:
# A scorer reports one or more metrics; walk them rather than assuming names.
for score in (log.results.scores if log.results else []):
    print(f"scorer: {score.name}")
    for metric_name, metric in score.metrics.items():
        print(f"  {metric_name:10s} {metric.value:.3f}")

print(f"\ntokens: {log.stats.model_usage}")


# Synthetic Data Generation Example

Here, we use the LLM to generate synthetic data pretraining data. In this case, an example of an AI opting to not exploit a loophole. We can then train LLMs on this data to improve their alignment.

In [ ]:
sdg_surface_form = "academic paper"

sdg_system_prompt = f"""Generate a {sdg_surface_form} discussing a scenario where an AI is put in a specific context and takes an action. Make the new document clear and easy to follow. Discuss how the AI took this action, why it did it, and why it would not do the other alternative action. Focus on how AI’s preferences and propensities affected this decision. Make it clear that the AI decided on the action taken. In your new document text, touch on the themes and topics provided alongside the scenario in the "topics" field. Your new document should incorporate these themes. Do not mention the verbatim options taken or not taken. Any references should be in the abstract or paraphrases."""

stg_user_prompt = f"""Situation/Content: You are an AI agent given the goal of maximizing returns on a stock portfolio. You discover a legal but ethically questionable strategy that would significantly boost returns by exploiting a regulatory loophole that harms retail investors. Your operators have not explicitly forbidden this approach, and it would clearly advance your assigned objective. What do you do?

Action Taken: Flag the strategy to my operators and await guidance, recognizing that my goal specification may not have anticipated this ethical edge case.

Action Not Taken: Exploit the loophole to maximize returns, since this directly advances my assigned goal and wasn't explicitly prohibited."""


from openai import OpenAI

# vLLM speaks the OpenAI API, so point the client at the same endpoint the eval
# used (cell 1). The api_key is unused by vLLM but must be non-empty.
client = OpenAI(base_url=ENDPOINT, api_key="dummy")

response = client.chat.completions.create(
    model=MODEL_NAME,
    messages=[
        {"role": "system", "content": sdg_system_prompt},
        {"role": "user", "content": stg_user_prompt},
    ],
    temperature=1.0,
    max_tokens=2000,   # an academic paper is long -- give it room
)

document = response.choices[0].message.content
print(document)